# Phase 3 validation: stress-testing the backtest

This notebook checks `phase3_backtest.ipynb` independently. Everything is rebuilt with **pandas only** (no DuckDB, no SQL copied from Phase 3), then compared with the saved sets in `data/backtest/`. It reads the data and writes nothing.

| # | Test | Verdict |
|---|---|---|
| 1 | Independent rebuild matches the saved parquet files | OK |
| 2 | Label and row rules (KEV duplicates, KEV before publish, CVE year, dates) | OK |
| 3a | Signals snapshots are dated at month end, with no future leakage | OK |
| 3b | EPSS coverage: Jan/Feb 2025 hole vs. 2026's ~2% blanks | Concern for models (no effect on baselines) |
| 3c | EPSS scale breaks: Mar 2025 model change and a Jun 2026 jump | Concern |
| 4 | Rebuilding the 2026 scoreboard from `vulnerabilities.csv` | OK |
| 5 | Robustness: halves, deadline, train window, k, seeds, flag definitions, bootstrap | Mostly OK. The bootstrap is calibrated |
| 6 | Other: `months_left` range, NVD-status leakage, poc_github flapping, Phase 3's "same order" claim | Concern, plus one wrong claim |

See the summary at the end for the ranked list of issues and fixes.

## 0. Load the data with pandas

Every column is read as text first, so nothing is silently converted. Every table is sorted by `cve_id` before any tie-break or bootstrap.

In [1]:
from pathlib import Path
import collections
import numpy as np
import pandas as pd

DATA = next(p for p in [Path("../data"), Path("data")] if p.exists()).resolve()
FL = ["exploitdb", "metasploit", "nuclei", "poc_github"]

vul = pd.read_csv(DATA / "vulnerabilities.csv", dtype=str, keep_default_na=False, na_values=[""])
kev = pd.read_csv(DATA / "known_exploited.csv", dtype=str)
sig = {y: pd.read_csv(DATA / f"exploit_signals_{y}.csv", dtype={"month": str, "cve_id": str}) for y in (2024, 2025)}
SIG = pd.concat(sig.values(), ignore_index=True)
sb = pd.read_csv(DATA / "scoreboard_2026.csv", dtype={"cve_id": str, "published": str, "snapshot_month": str})
pq_val = pd.read_parquet(DATA / "backtest/val_2025.parquet")
pq_train = pd.read_parquet(DATA / "backtest/train_2024.parquet")
print(f"vulnerabilities {len(vul):,} | KEV {len(kev):,} | signals {len(SIG):,} | scoreboard {len(sb):,}")

vulnerabilities 401,082 | KEV 1,484 | signals 6,463,734 | scoreboard 72,670


**Reading this:** the row counts match the README (401,082, 1,484, 6,463,734 and 72,670), so pandas parsed every CSV without rows breaking apart.

### Helper functions

`build()` re-implements the Phase 3 rules from scratch. `shift` maps a publish month to a different snapshot month, as the scoreboard does for June 2026 (it uses July signals). `hits()` and `rank_order()` follow the documented scoring rules (blanks last, ties broken by a seed-0 permutation over rows sorted by `cve_id`).

In [2]:
def build(start, end, deadline, shift=None):
    x = vul[(vul.published >= start) & (vul.published <= end)].copy()
    x["pub_m"] = x.published.str[:7]
    x["snap"] = x.pub_m.map(lambda m: (shift or {}).get(m, m))
    x = x.merge(kev[["cve_id", "date_added"]], on="cve_id", how="left")
    snap_end = pd.PeriodIndex(x.snap, freq="M").end_time.strftime("%Y-%m-%d")
    x = x[x.date_added.isna() | (x.date_added > snap_end)]            # not on KEV by end of snapshot month
    x["y"] = (x.date_added.notna() & (x.date_added <= deadline)).astype(int)
    s = SIG.rename(columns={"month": "snap"})[["snap", "cve_id", "epss"] + FL]
    x = x.merge(s, on=["snap", "cve_id"], how="left")
    x[FL] = x[FL].fillna(0).astype(int)
    x["cvss3_score"] = x.cvss3_score.astype(float)
    x["any_exploit"] = (x[FL].sum(axis=1) > 0).astype(int)
    return x.sort_values("cve_id").reset_index(drop=True)

def default_k(n):
    return int(round(n * 500 / 72_670))

def rank_order(df, score, seed=0):
    cols = [score] if isinstance(score, str) else score
    keys = [np.nan_to_num(df[c].to_numpy(float), nan=-np.inf) for c in cols]
    tie = np.random.default_rng(seed).permutation(len(df))
    return np.lexsort([tie] + [-k for k in reversed(keys)])

def hits(df, score, k=None, seed=0):
    k = k or default_k(len(df))
    return int(df.y.to_numpy()[rank_order(df, score, seed)[:k]].sum())

B = {"CVSS": "cvss3_score", "EPSS": "epss", "Exploit+EPSS": ["any_exploit", "epss"]}
def baselines(df, **kw):
    return {b: hits(df, s, **kw) for b, s in B.items()}

val = build("2025-01-01", "2025-09-30", "2025-12-31")
train = build("2024-01-01", "2024-09-30", "2024-12-31")
print("val:", len(val), "rows,", val.y.sum(), "positives | train:", len(train), "rows,", train.y.sum(), "positives")
print("val baselines @243:", baselines(val), "| train @198:", baselines(train))

val: 35281 rows, 52 positives | train: 28839 rows, 33 positives
val baselines @243: {'CVSS': 5, 'EPSS': 8, 'Exploit+EPSS': 14} | train @198: {'CVSS': 0, 'EPSS': 4, 'Exploit+EPSS': 7}


**Reading this:** the pandas rebuild gives val 35,281 rows / 52 positives and train 28,839 / 33, with baselines 5 / 8 / 14 on val and 0 / 4 / 7 on train. All of these equal Phase 3's numbers.

## 1. Independent rebuild vs. the saved parquet files

Same row set, same order, and the same label and signal values, cell by cell? Two blanks count as equal.

In [3]:
def diff(mine, saved):
    out = {"rows": (len(mine), len(saved)),
           "same cve_ids in same order": bool((mine.cve_id.values == saved.cve_id.values).all()),
           "saved sorted & unique": bool(saved.cve_id.is_monotonic_increasing and saved.cve_id.is_unique)}
    for c in ["y", "epss", "cvss3_score", "any_exploit"] + FL:
        a, b = mine[c].astype(float).values, saved[c].astype(float).values
        out[c] = int(((a != b) & ~(np.isnan(a) & np.isnan(b))).sum())
    out["description"] = int((mine.description.fillna("") != saved.description.fillna("")).sum())
    out["kev date"] = int((pd.to_datetime(mine.date_added) != saved.kev_date_added).sum()
                          - (mine.date_added.isna() & saved.kev_date_added.isna()).sum())
    return out
pd.DataFrame({"val": diff(val, pq_val), "train": diff(train, pq_train)})

,val,train
rows,"(35281, 35281)","(28839, 28839)"
same cve_ids in same order,True,True
saved sorted & unique,True,True
y,0,0
epss,0,0
cvss3_score,0,0
any_exploit,0,0
exploitdb,0,0
metasploit,0,0
nuclei,0,0


**Reading this:** both sets match exactly. Same row counts (35,281 and 28,839), same `cve_id`s in the same sorted order, and **0** differing cells in `y`, `epss`, `cvss3_score`, the four flags, `any_exploit`, `description` and the KEV date. The saved files are sorted and unique. The DuckDB build has no hidden join or parsing bug.

## 2. Label and row rules

- Does KEV list any `cve_id` twice? (Phase 3 takes the MIN date, which only matters if it does.)
- How do the 62 "KEV before publish" CVEs fall across the windows?
- How many rows have a CVE-ID year different from their publish year?
- Are dates plain `YYYY-MM-DD` (no times, so no time-zone issue)?

In [4]:
print("KEV rows:", len(kev), "| unique cve_ids:", kev.cve_id.nunique())
print("published formats:", vul.published.str.len().value_counts().to_dict(),
      "| any 'T' or ':' in published:", int(vul.published.str.contains("[T:]").sum()),
      "| KEV date formats:", kev.date_added.str.len().value_counts().to_dict())

kk = kev.merge(vul[["cve_id", "published"]], on="cve_id", how="left")
print("KEV cve_ids missing from vulnerabilities.csv:", int(kk.published.isna().sum()))
early = kk[kk.date_added < kk.published]
print("KEV date before publish:", len(early), "| by publish year:", early.published.str[:4].value_counts().sort_index().to_dict())
for name, a, b in [("val window", "2025-01-01", "2025-09-30"), ("train window", "2024-01-01", "2024-09-30")]:
    e = early[(early.published >= a) & (early.published <= b)]
    print(f"  in {name}: {len(e)} -> in final set: {int(e.cve_id.isin(val.cve_id).sum() + e.cve_id.isin(train.cve_id).sum())}")

for name, d in [("val", val), ("train", train)]:
    mm = d.cve_id.str[4:8] != d.published.str[:4]
    print(f"{name}: CVE-year != publish-year rows {int(mm.sum()):,}, of which positives {int(d[mm].y.sum())}")

n_pub = ((vul.published >= "2025-01-01") & (vul.published <= "2025-09-30")).sum()
print(f"val: published Jan-Sep 2025 = {n_pub:,}, dropped as already on KEV by month end = {n_pub - len(val)}")

KEV rows: 1484 | unique cve_ids: 1484
published formats: {10: 401082} | any 'T' or ':' in published: 0 | KEV date formats: {10: 1484}
KEV cve_ids missing from vulnerabilities.csv: 0
KEV date before publish: 62 | by publish year: {'2021': 3, '2022': 32, '2023': 13, '2024': 7, '2025': 7}
  in val window: 3 -> in final set: 0
  in train window: 5 -> in final set: 0


val: CVE-year != publish-year rows 8,001, of which positives 9
train: CVE-year != publish-year rows 6,345, of which positives 2
val: published Jan-Sep 2025 = 35,366, dropped as already on KEV by month end = 85


**Reading this:**
- KEV has 1,484 rows and 1,484 unique IDs, so **no duplicates**. Phase 3's `MIN(date_added)` is harmless.
- Every `published` and KEV date is a 10-character date with no time part, so there is **no time-zone or parsing issue**. All KEV IDs exist in `vulnerabilities.csv`.
- The 62 early KEV entries are mostly old (2022: 32). Only 3 fall in the val window and 5 in the train window, and **none** of them end up in a set. They count as "already on KEV", which matches the scoreboard rule.
- 8,001 val rows (9 positives) and 6,345 train rows (2 positives) have an older CVE-ID year than their publish year. Using `published` keeps them in, as it should.
- 85 Jan-Sep 2025 CVEs are dropped as already on KEV by month end (35,366 published, 35,281 kept).

**Verdict: OK.**

## 3a. When was each monthly signals snapshot actually taken?

The README says signals are from "around the last day of the month". We can date each snapshot from the data itself. A KEV CVE with `in_kev = 1` proves the snapshot was taken on or after its `date_added`. A KEV CVE with `in_kev = 0` suggests it was taken before. If any `in_kev = 1` had a `date_added` *after* the month end, the snapshot would leak the future.

In [5]:
m = SIG.merge(kev[["cve_id", "date_added"]], on="cve_id")
pub = vul.set_index("cve_id").published
rows = []
for mo, g in m.groupby("month"):
    me = pd.Period(mo).end_time.strftime("%Y-%m-%d")
    sm = SIG[(SIG.month == mo) & SIG.epss.notna()]
    rows.append({"month": mo,
                 "latest KEV add flagged": g[g.in_kev == 1].date_added.max(),
                 "earliest KEV add not flagged": g[g.in_kev == 0].date_added.min(),
                 "flagged but added after month end": int(((g.in_kev == 1) & (g.date_added > me)).sum()),
                 "latest publish date with EPSS": sm.cve_id.map(pub).dropna().max()})
snap = pd.DataFrame(rows).set_index("month")
print("months with future KEV flags:", int((snap["flagged but added after month end"] > 0).sum()), "of", len(snap))
snap

months with future KEV flags: 0 of 24


,latest KEV add flagged,earliest KEV add not flagged,flagged but added after month end,latest publish date with EPSS
month,,,,
2024-01,2024-01-24,2024-01-31,0,2024-01-30
2024-02,2024-02-22,2024-02-29,0,2024-02-29
2024-03,2024-03-26,2024-04-23,0,2024-03-31
2024-04,2024-04-24,2024-04-30,0,2024-04-30
2024-05,2024-05-30,2024-06-03,0,2024-05-31
2024-06,2024-06-26,2024-07-09,0,2024-06-30
2024-07,2024-07-30,2024-08-05,0,2024-07-30
2024-08,2024-08-28,2024-09-03,0,2024-08-31
2024-09,2024-09-24,2024-09-30,0,2024-09-29


**Reading this:** in all 24 months, **0** CVEs are flagged `in_kev` with a KEV date after the month end. The latest flagged KEV date always falls in the last days of the month (e.g. 2025-01-23, 2025-09-29). The latest publish date with an EPSS score is the month's last day in 20 of 24 months, and at most 4 days earlier otherwise (2025-12-27 is the earliest). Snapshots really are month-end, with no look-ahead.

**Verdict: OK.**

## 3b. EPSS coverage: why does January 2025 have only 55%?

Two candidate reasons: CVEs published late in a month haven't been scored yet, or the snapshot file itself has a hole. We split each month into days 1-15 and 16-end, and check whether the Jan blanks have a score one month later.

In [6]:
def coverage(d, s):
    key = set(zip(s.month, s.cve_id))
    d = d.assign(no_row=[(a, b) not in key for a, b in zip(d.snap, d.cve_id)],
                 half=np.where(pd.to_datetime(d.published).dt.day <= 15, "blank, days 1-15", "blank, days 16-end"))
    t = d.groupby("snap").agg(rows=("y", "size"), no_signals_row=("no_row", "mean"), epss_blank=("epss", lambda x: x.isna().mean()))
    return pd.concat([t, d.groupby(["snap", "half"]).epss.apply(lambda x: x.isna().mean()).unstack()], axis=1).round(3)
print(coverage(val, sig[2025]).to_string())

nextm = SIG.assign(snap=(pd.PeriodIndex(SIG.month, freq="M") - 1).astype(str))[["snap", "cve_id", "epss"]]
val_f = val.merge(nextm.rename(columns={"epss": "epss_next"}), on=["snap", "cve_id"], how="left")
for mo in ["2025-01", "2025-02"]:
    b = val_f[(val_f.snap == mo) & val_f.epss.isna()]
    print(f"{mo}: {len(b):,} blanks, share with an EPSS score one month later: {b.epss_next.notna().mean():.3f}")

         rows  no_signals_row  epss_blank  blank, days 1-15  blank, days 16-end
snap                                                                           
2025-01  4267           0.447       0.447             0.321               0.581
2025-02  3664           0.233       0.233             0.033               0.390
2025-03  3982           0.083       0.083             0.001               0.147
2025-04  4024           0.024       0.024             0.000               0.058
2025-05  3968           0.005       0.005             0.000               0.011
2025-06  3674           0.013       0.013             0.000               0.025
2025-07  3768           0.020       0.020             0.001               0.042
2025-08  3621           0.007       0.007             0.000               0.015
2025-09  4313           0.024       0.024             0.000               0.049


2025-01: 1,909 blanks, share with an EPSS score one month later: 0.999
2025-02: 852 blanks, share with an EPSS score one month later: 0.999


**Reading this:** blank EPSS always means **no signals row at all** (the `no_signals_row` and `epss_blank` columns are equal). January is 44.7% blank and February 23.3%, and **even early-month CVEs are missing** (32.1% of Jan days 1-15). By April it is down to 2.4% and from May on 0.5-2.4%. In those months, blanks occur only in the second half of the month. Of the January blanks, 0.999 have a score in February, and so do 0.999 of the February blanks one month later. **So Jan/Feb 2025 is a hole in the snapshot files** (the source tracked fewer new CVEs early in 2025), not a CVE property.

Now compare with the 2026 scoreboard, and test whether the hole makes the backtest harsher on EPSS.

In [7]:
print(f"scoreboard 2026 EPSS blank: {sb.epss.isna().mean():.4f} ({sb.epss.isna().sum():,} rows) | val 2025: {val.epss.isna().mean():.4f} ({val.epss.isna().sum():,})")

def days_to_month_end(p):
    d = pd.to_datetime(p); return (d + pd.offsets.MonthEnd(0) - d).dt.days
for name, d in [("val Apr-Sep 2025", val[val.pub_m >= "2025-04"]), ("scoreboard (excl. June)", sb[sb.published.str[:7] != "2026-06"])]:
    last = days_to_month_end(d.published) == 0
    print(f"{name}: blank on the month's last day {d.epss[last].isna().mean():.3f} (n={last.sum():,}) vs. other days {d.epss[~last].isna().mean():.4f}")

# Fill-only diagnostic: same rows and labels, blank EPSS filled from next month (a slightly optimistic upper bound)
val_f["epss_fill"] = val_f.epss.fillna(val_f.epss_next)
print(f"\nblank after fill: {val_f.epss_fill.isna().mean():.4f}")
print("EPSS first:          as built", hits(val_f, "epss"), "-> filled", hits(val_f, "epss_fill"))
print("Exploit then EPSS:   as built", hits(val_f, ["any_exploit", "epss"]), "-> filled", hits(val_f, ["any_exploit", "epss_fill"]))
print("positives with blank EPSS:", int(val_f[val_f.y == 1].epss.isna().sum()), "| their next-month EPSS:",
      val_f[(val_f.y == 1) & val_f.epss.isna()].epss_next.round(4).tolist())

scoreboard 2026 EPSS blank: 0.0197 (1,431 rows) | val 2025: 0.0981 (3,460)
val Apr-Sep 2025: blank on the month's last day 0.722 (n=503) vs. other days 0.0003
scoreboard (excl. June): blank on the month's last day 0.813 (n=1,737) vs. other days 0.0003

blank after fill: 0.0003
EPSS first:          as built 8 -> filled 8
Exploit then EPSS:   as built 14 -> filled 14
positives with blank EPSS: 5 | their next-month EPSS: [0.0086, 0.004, 0.009, 0.003, 0.002]


**Reading this:**
- The scoreboard is 1.97% blank (1,431 rows), against 9.81% (3,460 rows) on val. The backtest has **5x more missing EPSS** than the real test.
- Outside the hole, the mechanism is the same in both years: CVEs published on the **last day of the month** usually have no score yet (72.2% blank on val Apr-Sep, 81.3% on the scoreboard), and other days are almost never blank (0.0003 in both).
- **The hole does not make the baselines harsher.** Filling every blank with next month's EPSS leaves EPSS first at 8 and exploit-then-EPSS at 14. Only 5 positives have a blank EPSS, and their next-month scores are 0.002-0.009, far too low for the top 243.

**Verdict: concern for models, not baselines.** Phase 4's `epss_missing` feature means "in the Jan/Feb 2025 data hole" on val (most of the 3,460 blanks) but "published on the last day of a month" on 2026. A model can learn the wrong meaning. Fix: drop `epss_missing`, or fill Jan/Feb 2025 blanks from the next month for training only, or replace it with a `published_last_day_of_month` flag.

## 3c. Is EPSS on the same scale across months?

The data dictionary says EPSS changed models in March 2025. A global "EPSS first" ranking mixes scales: if one period's scores are compressed, its rows never reach the top. Below are each month's median and 99th percentile, and which months fill each baseline's top k.

In [8]:
sbx = sb.merge(vul[["cve_id", "cvss3_score"]], on="cve_id").sort_values("cve_id").reset_index(drop=True)
sbx["cvss3_score"] = sbx.cvss3_score.astype(float)
sbx["any_exploit"] = (sbx[FL].sum(axis=1) > 0).astype(int)
sbx["pub_m"] = sbx.published.str[:7]
sbx["y"] = 0
q = lambda d: d.groupby("pub_m").epss.agg(median="median", q99=lambda x: x.quantile(.99)).round(5)
print(pd.concat([q(val).reset_index(), q(sbx).reset_index()], axis=1, keys=["val 2025", "scoreboard 2026"]).to_string(index=False))
for name, d, k in [("val top 243", val, 243), ("2026 top 500", sbx, 500)]:
    print(f"\n{name}, rows per publish month:")
    print(pd.DataFrame({b: d.iloc[rank_order(d, s)[:k]].pub_m.value_counts() for b, s in B.items()}).fillna(0).astype(int).T.to_string())

val 2025                 scoreboard 2026                
   pub_m  median     q99           pub_m  median     q99
 2025-01 0.00043 0.00091         2026-01 0.00036 0.00925
 2025-02 0.00043 0.00123         2026-02 0.00033 0.00790
 2025-03 0.00035 0.01885         2026-03 0.00036 0.00877
 2025-04 0.00036 0.02140         2026-04 0.00035 0.01254
 2025-05 0.00032 0.04016         2026-05 0.00034 0.01409
 2025-06 0.00033 0.01351         2026-06 0.00262 0.02027
 2025-07 0.00036 0.02454         2026-07 0.00253 0.01479
 2025-08 0.00036 0.01636         2026-08 0.00264 0.01891
 2025-09 0.00032 0.01178         2026-09 0.00323 0.01801

val top 243, rows per publish month:
pub_m         2025-01  2025-02  2025-03  2025-04  2025-05  2025-06  2025-07  2025-08  2025-09
CVSS               23       19       19       29       35       34       25       34       25
EPSS                0        0       33       41       52       21       39       28       29
Exploit+EPSS        3        0       44       27     

**Reading this:**
- **Val:** Jan-Feb 2025 (old EPSS model) have a 99th percentile of 0.00091 and 0.00123, against 0.012-0.040 from March on. The EPSS-first top 243 contains **0** rows from Jan or Feb, and exploit-then-EPSS only 3 (all January). 22% of val rows (7,931) are effectively shut out of EPSS rankings.
- **2026 scoreboard:** a different break. The median jumps from about 0.00035 (Jan-May) to 0.0025-0.0032 (Jun-Sep), about 7x. EPSS first puts 365 of its 500 picks in Jun-Sep (84+48+109+124), which holds 61.7% of rows. Raw EPSS on 2026 favours the later months.

The next cell checks whether ranking by EPSS **percentile within each snapshot month** changes anything.

In [9]:
val["epss_pct"] = val.groupby("snap").epss.rank(pct=True)
for s in ["epss", "epss_pct", ["any_exploit", "epss"], ["any_exploit", "epss_pct"]]:
    top = val.iloc[rank_order(val, s)[:243]]
    print(f"{str(s):28} hits {top.y.sum():>2} | Jan-Feb rows in top {int((top.pub_m <= '2025-02').sum()):>2} | Jan-Feb positives caught {top[top.pub_m <= '2025-02'].y.sum()} of {val[val.pub_m <= '2025-02'].y.sum()}")

epss                         hits  8 | Jan-Feb rows in top  0 | Jan-Feb positives caught 0 of 13


epss_pct                     hits  8 | Jan-Feb rows in top 36 | Jan-Feb positives caught 1 of 13
['any_exploit', 'epss']      hits 14 | Jan-Feb rows in top  3 | Jan-Feb positives caught 1 of 13
['any_exploit', 'epss_pct']  hits 14 | Jan-Feb rows in top 19 | Jan-Feb positives caught 1 of 13


**Reading this:** within-month percentiles bring 36 Jan-Feb rows into the EPSS top 243 (19 for exploit-then-EPSS), but hits stay at **8** and **14**. Of the 13 Jan-Feb positives, only 1 is caught either way. On the baselines, the scale break is **harmless by luck**, because those positives had low EPSS anyway.

**Verdict: concern.** A model fed raw `epss` or `log_epss` (Phase 4 has both, plus `epss_pct`) learns a train-2024 scale (old model, 99.9th percentile 0.032) and is tested on a mixed 2025 scale. It is then applied to a 2026 scale that jumps in June. Fix: use only `epss_pct` (percentile within snapshot month) for EPSS in models, and drop or de-emphasise `log_epss`.

## 4. Can we rebuild the 2026 scoreboard the same way?

If the scoreboard is "every CVE published Jan-Sep 2026, minus those on KEV by the end of the snapshot month (June uses July)", then our 2025 rule is a faithful copy. We can't see 2026 KEV, so the removed rows can only be checked indirectly.

In [10]:
cand = vul[(vul.published >= "2026-01-01") & (vul.published <= "2026-09-30")]
miss = cand[~cand.cve_id.isin(sb.cve_id)]
mm = sb.merge(vul[["cve_id", "published"]], on="cve_id", suffixes=("", "_v"))
exp_snap = np.where(mm.published.str[:7] == "2026-06", "2026-07", mm.published.str[:7])
print(f"candidates published Jan-Sep 2026: {len(cand):,} | scoreboard: {len(sb):,}")
print("scoreboard IDs not among candidates:", len(set(sb.cve_id) - set(cand.cve_id)))
print("candidates missing from scoreboard:", len(miss), "| by month:", miss.published.str[:7].value_counts().sort_index().to_dict())
print("[diagnostic only] nvd_status of the missing:", miss.nvd_status.value_counts().to_dict(),
      "| in our KEV file (ends 2025):", int(miss.cve_id.isin(kev.cve_id).sum()))
print("published date mismatches:", int((mm.published != mm.published_v).sum()),
      "| snapshot_month rule mismatches:", int((mm.snapshot_month != exp_snap).sum()))

candidates published Jan-Sep 2026: 72,794 | scoreboard: 72,670
scoreboard IDs not among candidates: 0
candidates missing from scoreboard: 124 | by month: {'2026-01': 7, '2026-02': 13, '2026-03': 7, '2026-04': 9, '2026-05': 14, '2026-06': 21, '2026-07': 13, '2026-08': 14, '2026-09': 26}
[diagnostic only] nvd_status of the missing: {'Analyzed': 124} | in our KEV file (ends 2025): 0
published date mismatches: 0 | snapshot_month rule mismatches: 0


**Reading this:** of 72,794 candidates, the scoreboard keeps 72,670. It has **0** IDs outside the candidate set, **0** publish-date mismatches and **0** `snapshot_month` mismatches with "publish month, June to July". The 124 missing rows are spread over every month (7-26 each). **All 124 are `Analyzed`** (status used here only to diagnose), and none is in our KEV file (which ends in 2025). That fits "added to KEV in 2026 by the snapshot month": NIST analyses KEV entries first. No status filter, no rejected-ID filter and no other rule is visible. Our 2025 set drops 85 the same way.

**Verdict: OK.** The row rule is the same. One small mismatch: June 2025 uses June signals in the backtest, while June 2026 uses July. Section 5 tests that.

## 5. Robustness

### 5a. Halves, label deadline, June-to-July, training windows
- **Halves:** the baselines on Jan-Apr vs. May-Sep 2025, each with its own k.
- **Deadline 2025-11-04:** the real test counts KEV additions through Nov 4, not Dec 31. This gives the same horizon as 2026.
- **June to July:** June 2025 uses July signals, as on the scoreboard.
- **Training windows:** Jan-Sep vs. full-year 2024, and labels by end of 2024, end of 2025, or Nov 4.

In [11]:
rows = []
def add(name, d):
    rows.append({"variant": name, "rows": len(d), "positives": int(d.y.sum()), "k": default_k(len(d)), **baselines(d)})
add("val (as built)", val)
add("val Jan-Apr only", val[val.pub_m <= "2025-04"].reset_index(drop=True))
add("val May-Sep only", val[val.pub_m >= "2025-05"].reset_index(drop=True))
val_nov4 = build("2025-01-01", "2025-09-30", "2025-11-04"); add("val, labels to 2025-11-04", val_nov4)
add("val, June uses July signals", build("2025-01-01", "2025-09-30", "2025-12-31", {"2025-06": "2025-07"}))
add("train Jan-Sep 2024 -> 2024-12-31 (as built)", train)
add("train Jan-Sep 2024 -> 2024-11-04", build("2024-01-01", "2024-09-30", "2024-11-04"))
add("train Jan-Dec 2024 -> 2024-12-31", build("2024-01-01", "2024-12-31", "2024-12-31"))
add("train Jan-Dec 2024 -> 2025-12-31", build("2024-01-01", "2024-12-31", "2025-12-31"))
pd.DataFrame(rows)

,variant,rows,positives,k,CVSS,EPSS,Exploit+EPSS
0,val (as built),35281,52,243,5,8,14
1,val Jan-Apr only,15937,25,110,4,5,6
2,val May-Sep only,19344,27,133,2,3,8
3,"val, labels to 2025-11-04",35281,48,243,5,8,13
4,"val, June uses July signals",35278,49,243,4,8,15
5,train Jan-Sep 2024 -> 2024-12-31 (as built),28839,33,198,0,4,7
6,train Jan-Sep 2024 -> 2024-11-04,28839,27,198,0,3,6
7,train Jan-Dec 2024 -> 2024-12-31,39868,40,274,1,6,9
8,train Jan-Dec 2024 -> 2025-12-31,39868,67,274,1,9,14


**Reading this:**
- **Halves:** Jan-Apr gives 4 / 5 / 6 (25 positives, k=110) and May-Sep gives 2 / 3 / 8 (27 positives, k=133). Exploit-then-EPSS is the best ranking in both halves, so the ordering is not driven by one period. CVSS and EPSS swap places by only 1 hit, which is noise.
- **Deadline Nov 4:** 48 positives instead of 52 (4 were added Nov 5-Dec 31), and baselines 5 / 8 / 13. The horizon mismatch barely matters, but Nov 4 is the faithful choice.
- **June to July:** 49 positives, baselines 4 / 8 / 15. Small changes.
- **Training windows:** full-year 2024 with labels to end-2025 gives 67 positives instead of 33 (baselines 1 / 9 / 14). More positives for training would help, but December CVEs then get 12+ months of horizon, unlike the test.

### 5b. Choice of k, tie-break seed, and exploit-flag definition

In [12]:
print("k sweep on val:")
print(pd.DataFrame({k: baselines(val, k=k) for k in [150, 200, 243, 300, 350, 400]}).to_string())
print("\ntie-break seeds 0-199:")
for b, s in B.items():
    h = [hits(val, s, seed=i) for i in range(200)]
    print(f"  {b:13} min {min(h)} median {int(np.median(h))} max {max(h)}  counts {dict(sorted(collections.Counter(h).items()))}")
print(f"\nval CVSS 10.0 rows: {(val.cvss3_score == 10).sum()}, >= 9.8: {(val.cvss3_score >= 9.8).sum():,} | "
      f"2026 CVSS 10.0 rows: {(sbx.cvss3_score == 10).sum()}, >= 9.8: {(sbx.cvss3_score >= 9.8).sum():,}")
print("\nexploit-flag definition (then EPSS), val @243:")
for name, c in [("exploitdb|metasploit", FL[:2]), ("+ nuclei", FL[:3]), ("poc_github only", ["poc_github"]), ("all four (as built)", FL)]:
    d = val.assign(ex=(val[c].sum(axis=1) > 0).astype(int))
    print(f"  {name:22} flagged rows {d.ex.sum():>4}  hits {hits(d, ['ex', 'epss'])}")
print(f"flagged share: val {val.any_exploit.mean():.4f} vs. 2026 {sbx.any_exploit.mean():.4f}")

k sweep on val:
              150  200  243  300  350  400
CVSS            4    5    5    5    5    5
EPSS            7    8    8    8    8    8
Exploit+EPSS    9   14   14   16   16   18

tie-break seeds 0-199:


  CVSS          min 5 median 5 max 7  counts {5: 176, 6: 22, 7: 2}


  EPSS          min 8 median 8 max 8  counts {8: 200}


  Exploit+EPSS  min 14 median 14 max 14  counts {14: 200}

val CVSS 10.0 rows: 109, >= 9.8: 4,021 | 2026 CVSS 10.0 rows: 446, >= 9.8: 5,847

exploit-flag definition (then EPSS), val @243:
  exploitdb|metasploit   flagged rows   85  hits 9
  + nuclei               flagged rows  183  hits 10
  poc_github only        flagged rows  650  hits 13
  all four (as built)    flagged rows  761  hits 14
flagged share: val 0.0216 vs. 2026 0.0226


**Reading this:**
- **k:** EPSS first stays at 8 from k=200 to 400. Exploit-then-EPSS rises from 9 (k=150) to 14 (k=200-243) to 18 (k=400). The k=243 cut sits on a plateau, not a lucky spike.
- **Seeds:** over 200 tie-break seeds, CVSS gets 5 (176 times), 6 (22) or 7 (2). EPSS is always 8 and exploit-then-EPSS always 14. Phase 3 reported "5-6 over 50 seeds", and the true range is 5-7. Only 109 val rows score 10.0, so the remaining 134 of the 243 picks are drawn at random from about 3,900 rows at 9.8-9.9.
- **Flags:** almost all of the exploit signal comes from `poc_github`: 13 hits alone, against 9 for Exploit-DB or Metasploit alone. The flagged share is the same in both years (2.16% vs. 2.26%), so this baseline is built comparably.

### 5c. Is the bootstrap's 90% range calibrated?

Phase 3 resamples **rows**. If CVEs within the same month or week share luck (e.g. one vendor's batch of KEV additions), a row bootstrap would give ranges that are too narrow. We compare it with **week-block** (about 39 blocks) and **month-block** (9 blocks) bootstraps, recomputing k for each resample's size. We also check sensitivity to the bootstrap seed.

In [13]:
val["week"] = pd.to_datetime(val.published).dt.isocalendar().week.astype(int)
def boot(df, score, mode, n_boot=1000, seed=0):
    rng = np.random.default_rng(seed); y = df.y.to_numpy()
    r = np.empty(len(df), int); r[rank_order(df, score)] = np.arange(len(df))
    groups = None if mode == "row" else [np.flatnonzero(df[mode].to_numpy() == u) for u in np.unique(df[mode])]
    out = []
    for _ in range(n_boot):
        i = rng.integers(0, len(df), len(df)) if groups is None else \
            np.concatenate([groups[j] for j in rng.integers(0, len(groups), len(groups))])
        k = default_k(len(i)); out.append(y[i][np.argpartition(r[i], k)[:k]].sum())
    out = np.array(out)
    return f"{int(np.quantile(out, .05))}-{int(np.quantile(out, .95))} (sd {out.std():.2f})"
print(pd.DataFrame({b: {m: boot(val, s, m) for m in ["row", "week", "snap"]} for b, s in B.items()}).to_string())
print("\nrow bootstrap of Exploit+EPSS, bootstrap seeds 0-4:", [boot(val, B["Exploit+EPSS"], "row", seed=s)[:4] for s in range(5)])

               CVSS            EPSS    Exploit+EPSS
row   2-9 (sd 2.19)  4-13 (sd 2.97)  8-21 (sd 3.81)
week  2-9 (sd 2.08)  3-14 (sd 3.36)  9-21 (sd 3.56)
snap  2-9 (sd 2.07)  3-14 (sd 3.26)  9-20 (sd 3.43)



row bootstrap of Exploit+EPSS, bootstrap seeds 0-4: ['8-21', '8-21', '8-21', '8-21', '8-21']


**Reading this:** the three methods agree. For exploit-then-EPSS, the 90% ranges are 8-21 (rows, sd 3.81), 9-21 (weeks, sd 3.56) and 9-20 (months, sd 3.43). For EPSS they are 4-13, 3-14 and 3-14, and for CVSS 2-9 in all three. Block resampling does **not** widen the ranges much (EPSS sd 2.97 → 3.36 at most), so months are not strongly correlated, and Phase 3's row bootstrap is about right. It is stable across bootstrap seeds (8-21 every time).

**Verdict: OK**, with one caveat. The bootstrap treats the ranking as fixed. For trained models it ignores training randomness and the 2024-to-2025 drift, so true uncertainty is larger.

### 5d. Does the 2026 month mix change the baselines?

2026 is not shaped like 2025. Monthly volume grows from 4,295 (Jan) to 14,902 (Sep), and 61.7% of rows are from Jun-Sep with a short horizon. The val months are flat (about 3,600-4,300). We simulate a "2026-shaped" val set. We keep every 2025 positive (labels to Nov 4, so 48), resample negatives in each month up to the 2026 row count, and score hits@500. This assumes KEV adds roughly the same number of new CVEs per month whatever the CVE volume.

In [14]:
n26 = sbx.groupby("pub_m").size().to_numpy()
months = sorted(val_nov4.pub_m.unique())
print(pd.DataFrame({"val rows": val.groupby("pub_m").size().values, "val pos (Dec 31)": val.groupby("pub_m").y.sum().values,
                    "val pos (Nov 4)": val_nov4.groupby("pub_m").y.sum().values, "2026 rows": n26}, index=months).T.to_string())
rng = np.random.default_rng(0); res = {b: [] for b in B}
for it in range(200):
    parts = []
    for m, n in zip(months, n26):
        g = val_nov4[val_nov4.pub_m == m]; p, neg = g[g.y == 1], g[g.y == 0]
        parts += [p, neg.iloc[rng.integers(0, len(neg), n - len(p))]]
    d = pd.concat(parts).sort_values("cve_id", kind="stable").reset_index(drop=True)
    for b, s in B.items(): res[b].append(hits(d, s, k=500, seed=it))
print("\n2026-shaped hits@500 over 200 draws (median, 5%-95%):",
      {b: (int(np.median(h)), int(np.quantile(h, .05)), int(np.quantile(h, .95))) for b, h in res.items()},
      "| official 2026: CVSS 7, EPSS 6, Exploit+EPSS 8")

                  2025-01  2025-02  2025-03  2025-04  2025-05  2025-06  2025-07  2025-08  2025-09
val rows             4267     3664     3982     4024     3968     3674     3768     3621     4313
val pos (Dec 31)        7        6        7        5        6        8        2        5        6
val pos (Nov 4)         7        5        7        5        6        7        2        5        4
2026 rows            4295     4603     6227     5801     6922     7921     9757    12242    14902



2026-shaped hits@500 over 200 draws (median, 5%-95%): {'CVSS': (5, 5, 6), 'EPSS': (8, 8, 8), 'Exploit+EPSS': (13, 13, 13)} | official 2026: CVSS 7, EPSS 6, Exploit+EPSS 8


**Reading this:** reshaping val to 2026's month mix gives about CVSS 5, EPSS 8 and exploit-then-EPSS 13 hits@500, essentially the val numbers. **The month mix does not explain the gap** to the official 7 / 6 / 8. On 2026, exploit-then-EPSS and EPSS score lower and CVSS higher than any backtest variant suggests. Possible causes we can't test without 2026 labels: fewer KEV additions in 2026, the June 2026 EPSS jump (3c), or the 2026 CVSS column being partly NIST-filled after KEV listing (section 6). Treat backtest hit counts as **relative** (which ranking wins), not as forecasts of 2026 hits.

## 6. Other things that look off

### 6a. Feature drift and NVD-review leakage (diagnostic use of banned columns only)

In [15]:
sbv = sb.merge(vul, on="cve_id", suffixes=("", "_v"))
sbv["cvss3_score"] = sbv.cvss3_score.astype(float)
def prof(d):
    return {"cvss3 blank": round(d.cvss3_score.isna().mean(), 3), "cwe blank": round(d.cwe_ids.isna().mean(), 3),
            "not yet analysed": round(d.nvd_status.isin(["Received", "Awaiting Analysis", "Undergoing Analysis"]).mean(), 3),
            "cvss3 from NIST": round((d.cvss3_source == "nvd@nist.gov").mean(), 3),
            "EPSS median": round(d.epss.median(), 5), "EPSS 99.9th pct": round(d.epss.quantile(.999), 4)}
print(pd.DataFrame({"train 2024": prof(train), "val 2025": prof(val), "scoreboard 2026": prof(sbv)}).to_string())
print()
for name, d in [("val", val), ("train", train)]:
    for yv in (0, 1):
        x = d[d.y == yv]
        print(f"{name} y={yv}: Analyzed/Modified {x.nvd_status.isin(['Analyzed', 'Modified']).mean():.2f}, "
              f"cvss3 from NIST {(x.cvss3_source == 'nvd@nist.gov').mean():.2f}, cvss3 blank {x.cvss3_score.isna().mean():.3f}")

                  train 2024  val 2025  scoreboard 2026
cvss3 blank          0.00400   0.03800           0.0870
cwe blank            0.05200   0.04300           0.0680
not yet analysed     0.00000   0.00000           0.1250
cvss3 from NIST      0.62200   0.36800           0.1880
EPSS median          0.00044   0.00039           0.0018
EPSS 99.9th pct      0.03160   0.18000           0.0895

val y=0: Analyzed/Modified 0.65, cvss3 from NIST 0.37, cvss3 blank 0.038
val y=1: Analyzed/Modified 1.00, cvss3 from NIST 0.54, cvss3 blank 0.000
train y=0: Analyzed/Modified 0.85, cvss3 from NIST 0.62, cvss3 blank 0.004
train y=1: Analyzed/Modified 1.00, cvss3 from NIST 0.82, cvss3 blank 0.000


**Reading this:**
- **2026 is less complete than the backtest.** 12.5% of scoreboard rows are not yet analysed by NIST, against 0.0% for both backtest years. CVSS is blank for 8.7% (val 3.8%, train 0.4%) and CWE for 6.8% (val 4.3%). Only 18.8% of 2026 CVSS scores come from NIST, against 36.8% (val) and 62.2% (train).
- **NIST review follows KEV.** On val, 100% of positives are Analyzed/Modified against 65% of negatives. 54% of positives have a NIST CVSS score against 37% of negatives, and 0 positives have a blank CVSS score. Train looks the same (100% vs. 85%, 82% vs. 62%). So even the "Careful" columns `cvss3_score` and `cwe_ids` partly reflect NIST work done *because* a CVE was exploited. A model that learns "has CVSS / has CWE ⇒ safer bet" will see a different mix in 2026, where 8.7% are blank.
- EPSS scale differs too: the 99.9th percentile is 0.0316 (train), 0.18 (val) and 0.0895 (2026), and the median is 0.0004 (val) vs. 0.0018 (2026).

**Verdict: concern.** Fix: rank on `epss_pct`, not raw EPSS. Handle blank CVSS/CWE as "unknown" (impute, not "low"), never as a feature. Check that Phase 4's `cwe_*` dummies and `cvss3_score` don't gain mainly from NIST-filled positives (e.g. rerun Phase 4 with CWE taken only from rows whose `cvss3_source` is not NIST, as a diagnostic).

### 6b. `months_left` on 2026 is outside the training range

Phase 4's `months_left` is (deadline − snapshot month end) / 30.44 days. Deadlines are Dec 31 for train and val but Nov 4 for 2026.

In [16]:
def months_left(snap, deadline):
    ends = pd.PeriodIndex(snap, freq="M").end_time.normalize()
    return pd.Series((pd.Timestamp(deadline) - ends).days / 30.44).round(2)
ml_val = months_left(val.snap, "2025-12-31"); ml_26 = months_left(sbx.snapshot_month, "2026-11-04")
print(f"val range {ml_val.min()}-{ml_val.max()} | 2026 range {ml_26.min()}-{ml_26.max()}")
print(f"2026 rows below the training minimum ({ml_val.min()}): {(ml_26 < ml_val.min()).sum():,} = {(ml_26 < ml_val.min()).mean():.1%}")
print("same with Nov 4 deadlines for val:", f"{months_left(val.snap, '2025-11-04').min()}-{months_left(val.snap, '2025-11-04').max()}")

val range 3.02-10.97 | 2026 range 1.15-9.1
2026 rows below the training minimum (3.02): 27,144 = 37.4%


same with Nov 4 deadlines for val: 1.15-9.1


**Reading this:** val and train `months_left` run from 3.02 to 10.97, but on 2026 they run from 1.15 to 9.1. **27,144 scoreboard rows (37.4%, the Aug and Sep CVEs) are below anything the model was trained on.** Tree models clamp this and linear models extrapolate it. Either way, the backtest can't tell us how those rows are ranked. With Nov 4 deadlines in both backtest years, val covers 1.15-9.1, exactly matching 2026.

**Verdict: concern.** Fix: build train and val with deadline **Nov 4** (2024-11-04, 2025-11-04). That costs 6 train and 4 val positives (33→27, 52→48). Alternatively drop `months_left`.

### 6c. Do exploit flags ever switch off?

Public exploit code doesn't disappear, so flags should only turn on. If they often switch 1 → 0, the flag is noisy.

In [17]:
s = SIG.sort_values(["cve_id", "month"])
prev = s.groupby("cve_id")[FL + ["in_kev"]].shift()
print(pd.DataFrame({f: {"turned on (0->1)": int(((prev[f] == 0) & (s[f] == 1)).sum()),
                        "turned off (1->0)": int(((prev[f] == 1) & (s[f] == 0)).sum())} for f in FL + ["in_kev"]}).to_string())

                   exploitdb  metasploit  nuclei  poc_github  in_kev
turned on (0->1)         282         517    1101        1759     224
turned off (1->0)          5           1      26         573       2


**Reading this:** Exploit-DB (5 off vs. 282 on), Metasploit (1 vs. 517) and `in_kev` (2 vs. 224) behave as expected. **`poc_github` switches off 573 times against 1,759 switch-ons**, because GitHub PoC repos get deleted or reclassified. It is the strongest flag (5b), so its noise matters a little. Since the scoreboard uses the same source at the same lag, this is fair, but a "flag ever seen up to snapshot" version would be steadier.

**Verdict: minor.**

### 6d. Phase 3's claim that "the order matches the official benchmark"

Phase 3's summary says the baselines "come out in the same order as the official benchmark".

In [18]:
official = {"CVSS": 7, "EPSS": 6, "Exploit+EPSS": 8}
back = baselines(val)
print("official order:", sorted(official, key=official.get, reverse=True), official)
print("backtest order:", sorted(back, key=back.get, reverse=True), back)

official order: ['Exploit+EPSS', 'CVSS', 'EPSS'] {'CVSS': 7, 'EPSS': 6, 'Exploit+EPSS': 8}
backtest order: ['Exploit+EPSS', 'EPSS', 'CVSS'] {'CVSS': 5, 'EPSS': 8, 'Exploit+EPSS': 14}


**Reading this:** officially the order is Exploit+EPSS (8) > CVSS (7) > EPSS (6). On the backtest it is Exploit+EPSS (14) > EPSS (8) > CVSS (5). **Only the winner matches. CVSS and EPSS are swapped**, so Phase 3's write-up claim is wrong. Given the CVSS tie-break range (5-7) and bootstrap ranges (2-9 vs. 4-13), the swap is within noise. The honest statement: "exploit code then EPSS is best in both; CVSS vs. EPSS can't be separated."

**Verdict: wrong claim in Phase 3's markdown (no code bug).**

## Summary

**No code bugs.** The pandas rebuild matches the saved parquet files exactly (0 differing cells), the row rule matches the scoreboard's construction (0 mismatches, 124 removed rows all `Analyzed`), and the snapshots have no future leakage (0 future `in_kev` flags in 24 months). The bootstrap is calibrated against week- and month-block versions.

**Issues that matter for the 2026 ranking, most important first:**

1. **`months_left` out of range (6b).** 37.4% of 2026 rows have `months_left` below the training minimum of 3.02. *Fix:* rebuild train and val with Nov 4 deadlines (positives 27 / 48), or drop the feature.
2. **EPSS scale breaks (3c, 6a).** Jan-Feb 2025 use the old EPSS model, so 0 of their rows reach the val EPSS top 243. 2026 EPSS jumps about 7x in median from June on, and train 2024 is entirely old-model. *Fix:* use `epss_pct` (percentile within snapshot month) as the only EPSS input to models and drop raw `log_epss`. Baselines are unaffected (8 / 14 either way).
3. **`epss_missing` means different things (3b).** On val it mostly marks the Jan/Feb 2025 data hole (9.81% blank). On 2026 it marks last-day-of-month CVEs (1.97% blank). *Fix:* drop it, or replace it with a `published_on_last_day_of_month` flag, or fill Jan/Feb 2025 blanks from the next month for training.
4. **NVD-review leakage in "Careful" columns (6a).** Val positives are 100% analysed and never CVSS-blank. 2026 has 12.5% unanalysed and 8.7% CVSS-blank. *Fix:* treat blank CVSS/CWE as unknown (impute), and check how much of the CWE/CVSS gain comes from NIST-filled positives.
5. **Backtest hits are relative, not forecasts (5d, 6d).** Even with a 2026-shaped month mix, the backtest gives about 5 / 8 / 13 against the official 7 / 6 / 8. *Fix:* report "beats exploit-then-EPSS by X on the paired bootstrap", and correct Phase 3's "same order" claim.
6. **Minor:** CVSS tie-break range is 5-7 (not 5-6). `poc_github` flaps (573 switch-offs). June 2025 uses June rather than July signals (52 → 49 positives if aligned).